# اليوم الثالث — التعلم الخاضع للإشراف
نقيّم التصنيف، نبني الانحدار، ثم نقارن نموذجين على بيانات تحقق مستقلة.

## 0) تحميل بيانات منافذ تلقائيًا

In [ ]:
import numpy as np
import pandas as pd
DATA_URL = 'https://raw.githubusercontent.com/hanenalmayouf/FAML/main/public/data/manafeth_customers.csv'
df = pd.read_csv(DATA_URL)
print('تم تحميل بيانات منافذ ✅', df.shape)
display(df.head(3))

# المحطة 1 — ابنِ مسار التصنيف

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
drop_cols = ['customer_id','churned_30d','refund_issued','support_ticket_after_snapshot','next_month_orders','signup_date','snapshot_date']
X = df.drop(columns=drop_cols, errors='ignore')
y = df['churned_30d'].copy()
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
numeric = X.select_dtypes(include='number').columns.tolist()
categorical = X.select_dtypes(exclude='number').columns.tolist()
num_pipe = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))])
prep = ColumnTransformer([('num', num_pipe, numeric), ('cat', cat_pipe, categorical)])
logistic = Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=1000))])
logistic.fit(X_train, y_train)
val_probability = logistic.predict_proba(X_val)[:, 1]
print('نموذج التصنيف جاهز ✅')

# المحطة 2 — غيّر العتبة وشاهد الأخطاء

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score
rows = []
for threshold in [0.30, 0.50, 0.70]:
    prediction = (val_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, prediction).ravel()
    rows.append({'العتبة':threshold,'TP':tp,'FP':fp,'FN':fn,'TN':tn,'Precision':precision_score(y_val,prediction,zero_division=0),'Recall':recall_score(y_val,prediction,zero_division=0)})
display(pd.DataFrame(rows).round(3))

# المحطة 3 — ابنِ نموذج انحدار

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
reg_drop = ['customer_id','avg_basket_sar','churned_30d','refund_issued','support_ticket_after_snapshot','next_month_orders','signup_date','snapshot_date']
X_reg = df.drop(columns=reg_drop, errors='ignore')
y_reg = df['avg_basket_sar'].copy()
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)
reg_numeric = X_reg.select_dtypes(include='number').columns.tolist()
reg_categorical = X_reg.select_dtypes(exclude='number').columns.tolist()
reg_prep = ColumnTransformer([('num', num_pipe, reg_numeric), ('cat', cat_pipe, reg_categorical)])
baseline = Pipeline([('prep', clone(reg_prep)), ('model', DummyRegressor(strategy='median'))])
linear = Pipeline([('prep', clone(reg_prep)), ('model', LinearRegression())])
for name, model in {'Baseline':baseline, 'Linear Regression':linear}.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    print(name, 'MAE=', round(mean_absolute_error(yr_test,pred),2), 'RMSE=', round(mean_squared_error(yr_test,pred,squared=False),2), 'R²=', round(r2_score(yr_test,pred),3))

# المحطة 4 — قارن نموذجين على بيانات التحقق

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
models = {'Logistic Regression': LogisticRegression(max_iter=1000), 'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5, random_state=42)}
results = []
trained = {}
for name, estimator in models.items():
    model = Pipeline([('prep', clone(prep)), ('model', estimator)])
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_val)[:, 1]
    results.append({'النموذج':name, 'Validation PR-AUC':average_precision_score(y_val, probability)})
    trained[name] = model
display(pd.DataFrame(results).round(3))

# المحطة 5 — افتح الاختبار مرة واحدة

In [ ]:
chosen_name = 'Logistic Regression'
chosen_model = trained[chosen_name]
test_probability = chosen_model.predict_proba(X_test)[:, 1]
print('Test PR-AUC:', round(average_precision_score(y_test, test_probability), 3))

# التسليم الختامي
جدول أثر العتبات، نتائج الانحدار، مقارنة النموذجين على Validation Set، ونتيجة الاختبار النهائي.